In [ ]:
import itertools
import numpy as np
import pandas as pd

def load_v_from_csv(csv_path: str, id_col: str = "case_id", v_col: str = "v_i"):
    """
    Charge v_i depuis un CSV et renvoie:
      - ids: liste des case_id dans l'ordre
      - v: numpy array (N,)
      - df: dataframe original trié
    """
    n_use = 24

    df_big = pd.read_csv(csv_path)
    df = df_big.sort_values("v_i", ascending=False).head(n_use).reset_index(drop=True)

    if id_col not in df.columns:
        raise ValueError(f"Colonne '{id_col}' introuvable. Colonnes disponibles: {list(df.columns)}")
    if v_col not in df.columns:
        raise ValueError(f"Colonne '{v_col}' introuvable. Colonnes disponibles: {list(df.columns)}")

    df = df[[id_col, v_col]].dropna().copy()
    df[id_col] = df[id_col].astype(int)
    df = df.sort_values(id_col).reset_index(drop=True)

    ids = df[id_col].to_numpy()
    v = df[v_col].to_numpy(dtype=float)
    return ids, v, df

def load_W_npy(npy_path: str) -> np.ndarray:
    W = np.load(npy_path)
    if W.ndim != 2 or W.shape[0] != W.shape[1]:
        raise ValueError(f"W doit être carrée. Shape trouvée: {W.shape}")
    return W.astype(float)

def load_W_csv_matrix(csv_path: str, ids: np.ndarray = None) -> np.ndarray:
    """
    Charge W depuis un CSV matrice.
    Deux cas:
      - CSV avec index/colonnes = case_id (recommandé) -> on réordonne selon ids si fourni
      - CSV sans labels (juste NxN) -> on lit brut
    """
    dfW = pd.read_csv(csv_path, index_col=0)

    try:
        dfW.index = dfW.index.astype(int)
        dfW.columns = dfW.columns.astype(int)
        has_labels = True
    except Exception:
        has_labels = False

    if ids is not None and has_labels:
        missing = set(ids) - set(dfW.index)
        if missing:
            raise ValueError(f"W ne contient pas tous les ids. Manquants (ex): {list(missing)[:10]}")
        dfW = dfW.loc[ids, ids]

    W = dfW.to_numpy(dtype=float)
    if W.shape[0] != W.shape[1]:
        raise ValueError(f"W doit être carrée. Shape trouvée: {W.shape}")
    return W

def brute_force_best_k(v: np.ndarray, W: np.ndarray, k: int = 5):
    """
    Brute force exact:
      max sum v_i + sum_{i<j} W_ij
      s.t. |S| = k
    Retourne (best_value, best_indices)
    """
    v = np.asarray(v, dtype=float)
    W = np.asarray(W, dtype=float)
    n = len(v)
    if W.shape != (n, n):
        raise ValueError(f"Incompatibilité dimensions: len(v)={n} mais W.shape={W.shape}")
    if not (0 <= k <= n):
        raise ValueError("k doit être entre 0 et n")

    W_ut = np.triu(W, k=1)

    best_val = -np.inf
    best_set = None

    for S in itertools.combinations(range(n), k):
        S_list = list(S)
        val = float(v[S_list].sum())
        val += float(W_ut[np.ix_(S_list, S_list)].sum())

        if val > best_val:
            best_val = val
            best_set = S

    return best_val, best_set

def main():
    V_CSV_PATH = "underwriting_dataset.csv"    
    ID_COL = "case_id"
    V_COL = "v_i"
    K = 5

    W_NPY_PATH = "W.npy"           
    W_CSV_PATH = "W.npy"        

    ids, v, _ = load_v_from_csv(V_CSV_PATH, id_col=ID_COL, v_col=V_COL)

    if W_NPY_PATH is not None:
        W = load_W_npy(W_NPY_PATH)
    else:
        W = load_W_csv_matrix(W_CSV_PATH, ids=ids)

    W = 0.5 * (W + W.T)
    np.fill_diagonal(W, 0.0)

    best_val, best_set = brute_force_best_k(v, W, k=K)
    best_set = list(best_set)

    best_case_ids = ids[best_set]
    best_v_sum = float(v[best_set].sum())
    best_w_sum = float(np.triu(W[np.ix_(best_set, best_set)], 1).sum())

    print("=== Résultat brute force exact ===")
    print(f"k = {K}, N = {len(ids)}")
    print(f"Valeur objectif totale : {best_val:.6f}")
    print(f"  - somme v_i          : {best_v_sum:.6f}")
    print(f"  - somme w_ij (i<j)   : {best_w_sum:.6f}")
    print("Dossiers sélectionnés (case_id) :", list(map(int, best_case_ids)))

    x = np.zeros(len(ids), dtype=int)
    x[best_set] = 1

    out = pd.DataFrame({"case_id": ids, "x_selected": x, "v_i": v})
    out.to_csv("best_selection.csv", index=False)
    print("Sauvegardé dans: best_selection.csv")


In [25]:
if __name__ == "__main__":
    main()

=== Résultat brute force exact ===
k = 5, N = 24
Valeur objectif totale : 2065.358024
  - somme v_i          : 1222.517371
  - somme w_ij (i<j)   : 842.840653
Dossiers sélectionnés (case_id) : [0, 2, 3, 8, 10]
Sauvegardé dans: best_selection.csv


In [ ]:
import numpy as np
import pandas as pd
import os

# =============================
# LOADING DATA
# =============================

def load_data(v_csv_path, w_path):
    df = pd.read_csv(v_csv_path)
    case_ids = df["case_id"].to_numpy()
    v = df["v_i"].to_numpy(dtype=float)

    # Load W
    if w_path.endswith(".npy"):
        W = np.load(w_path)
    else:
        W_df = pd.read_csv(w_path, index_col=0)
        W_df.index = W_df.index.astype(int)
        W_df.columns = W_df.columns.astype(int)
        W_df = W_df.loc[case_ids, case_ids]
        W = W_df.to_numpy(dtype=float)

    W = 0.5 * (W + W.T)
    np.fill_diagonal(W, 0.0)

    return case_ids, v, W


# =============================
# OBJECTIVE
# =============================

def objective(S, v, W):
    S = np.array(S)
    val = v[S].sum()
    if len(S) > 1:
        val += np.triu(W[np.ix_(S, S)], 1).sum()
    return float(val)


# =============================
# SIMPLE TABU SEARCH
# =============================

def tabu_search(v, W, k=5, max_iters=3, tabu_tenure=1):
    n = len(v)

    # Initial solution: top k v_i
    S = list(np.argsort(-v)[:k])
    best_S = S.copy()
    best_val = objective(S, v, W)
    current_val = best_val

    tabu = {}

    for it in range(max_iters):
        best_move = None
        best_delta = -np.inf

        for a in S:
            for b in range(n):
                if b in S:
                    continue

                # Compute delta
                remove_loss = v[a] + sum(W[a, j] for j in S if j != a)
                add_gain = v[b] + sum(W[b, j] for j in S if j != a)
                delta = add_gain - remove_loss

                move = (a, b)

                if move in tabu and tabu[move] > it:
                    continue

                if delta > best_delta:
                    best_delta = delta
                    best_move = move

        if best_move is None:
            break

        a, b = best_move

        # Apply move
        S.remove(a)
        S.append(b)
        current_val += best_delta

        # Update tabu
        tabu[(b, a)] = it + tabu_tenure  # forbid immediate reverse

        # Update best
        if current_val > best_val:
            best_val = current_val
            best_S = S.copy()

    return best_S, best_val


# =============================
# MAIN
# =============================

if __name__ == "__main__":

    V_CSV = "underwriting_dataset.csv"
    W_PATH = "W.npy"  

    case_ids, v, W = load_data(V_CSV, W_PATH)

    best_idx, best_val = tabu_search(v, W, k=5)

    selected_case_ids = case_ids[best_idx]

    print("Selected case_ids:", list(selected_case_ids))
    print("Total gain:", best_val)

Selected case_ids: [np.int64(10), np.int64(3), np.int64(0), np.int64(2), np.int64(8)]
Total gain: 2065.358023711785
